#### Faiss
Facebook AI Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [12]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("speech.txt")
document = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=200,chunk_overlap=30)
docs = text_splitter.split_documents(document)

Created a chunk of size 1147, which is longer than the specified 200
Created a chunk of size 1141, which is longer than the specified 200


In [13]:
embeddings =(
    OllamaEmbeddings(model="llama3.2:latest") ## By default it use llama2
)
db=FAISS.from_documents(docs, embeddings)
db

In [14]:
### querying 
query="Who were the parents of Chhatrapati Shivaji Maharaj?"
docs=db.similarity_search(query)
docs[0].page_content

'Chhatrapati Shivaji Maharaj (1630–1680) was the visionary founder of the Maratha Empire and a defining figure in Indian\n history whose legacy of "Hindavi Swarajya" (self-rule) transformed the socio-political landscape of the 17th century.\n  Born at Shivneri Fort to Shahaji Bhonsle and Jijabai, his character was profoundly shaped by his mother’s teachings of\n   justice, valor, and the stories of the Ramayana and Mahabharata. Growing up in a period dominated by the Mughal Empire\n    and the Deccan Sultanates, Shivaji displayed exceptional military and leadership qualities from a young age. At just 16,\n     he captured the Torna Fort, marking the beginning of a relentless campaign to liberate his people from foreign oppression.\n      Over the following decades, he utilized innovative guerrilla warfare tactics, known as Ganimi Kava, to outmaneuver the\n       much larger and more powerful armies of the Mughals and the Bijapur Sultanate. His strategic brilliance was immortalized\n   

In [17]:
query="fort captured by Shivaji at the age of 16?"
docs=db.similarity_search(query)
docs[0].page_content

'Chhatrapati Shivaji Maharaj (1630–1680) was the visionary founder of the Maratha Empire and a defining figure in Indian\n history whose legacy of "Hindavi Swarajya" (self-rule) transformed the socio-political landscape of the 17th century.\n  Born at Shivneri Fort to Shahaji Bhonsle and Jijabai, his character was profoundly shaped by his mother’s teachings of\n   justice, valor, and the stories of the Ramayana and Mahabharata. Growing up in a period dominated by the Mughal Empire\n    and the Deccan Sultanates, Shivaji displayed exceptional military and leadership qualities from a young age. At just 16,\n     he captured the Torna Fort, marking the beginning of a relentless campaign to liberate his people from foreign oppression.\n      Over the following decades, he utilized innovative guerrilla warfare tactics, known as Ganimi Kava, to outmaneuver the\n       much larger and more powerful armies of the Mughals and the Bijapur Sultanate. His strategic brilliance was immortalized\n   

#### As a Retriever
We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers

In [18]:
retriver = db.as_retriever()
docs =retriver.invoke(query)
docs[0].page_content

'Chhatrapati Shivaji Maharaj (1630–1680) was the visionary founder of the Maratha Empire and a defining figure in Indian\n history whose legacy of "Hindavi Swarajya" (self-rule) transformed the socio-political landscape of the 17th century.\n  Born at Shivneri Fort to Shahaji Bhonsle and Jijabai, his character was profoundly shaped by his mother’s teachings of\n   justice, valor, and the stories of the Ramayana and Mahabharata. Growing up in a period dominated by the Mughal Empire\n    and the Deccan Sultanates, Shivaji displayed exceptional military and leadership qualities from a young age. At just 16,\n     he captured the Torna Fort, marking the beginning of a relentless campaign to liberate his people from foreign oppression.\n      Over the following decades, he utilized innovative guerrilla warfare tactics, known as Ganimi Kava, to outmaneuver the\n       much larger and more powerful armies of the Mughals and the Bijapur Sultanate. His strategic brilliance was immortalized\n   

#### Similarity Search with score
There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [19]:
docs_and_score=db.similarity_search_with_score(query)
docs_and_score

[(Document(id='e46c4e43-89dc-4211-be4b-07ad1f12a091', metadata={'source': 'speech.txt'}, page_content='Chhatrapati Shivaji Maharaj (1630–1680) was the visionary founder of the Maratha Empire and a defining figure in Indian\n history whose legacy of "Hindavi Swarajya" (self-rule) transformed the socio-political landscape of the 17th century.\n  Born at Shivneri Fort to Shahaji Bhonsle and Jijabai, his character was profoundly shaped by his mother’s teachings of\n   justice, valor, and the stories of the Ramayana and Mahabharata. Growing up in a period dominated by the Mughal Empire\n    and the Deccan Sultanates, Shivaji displayed exceptional military and leadership qualities from a young age. At just 16,\n     he captured the Torna Fort, marking the beginning of a relentless campaign to liberate his people from foreign oppression.\n      Over the following decades, he utilized innovative guerrilla warfare tactics, known as Ganimi Kava, to outmaneuver the\n       much larger and more po

In [20]:
### Saving And Loading
db.save_local("faiss_index")

In [21]:
new_db=FAISS.load_local("faiss_index",embeddings,allow_dangerous_deserialization=True)
docs=new_db.similarity_search(query)

In [22]:
docs

[Document(id='e46c4e43-89dc-4211-be4b-07ad1f12a091', metadata={'source': 'speech.txt'}, page_content='Chhatrapati Shivaji Maharaj (1630–1680) was the visionary founder of the Maratha Empire and a defining figure in Indian\n history whose legacy of "Hindavi Swarajya" (self-rule) transformed the socio-political landscape of the 17th century.\n  Born at Shivneri Fort to Shahaji Bhonsle and Jijabai, his character was profoundly shaped by his mother’s teachings of\n   justice, valor, and the stories of the Ramayana and Mahabharata. Growing up in a period dominated by the Mughal Empire\n    and the Deccan Sultanates, Shivaji displayed exceptional military and leadership qualities from a young age. At just 16,\n     he captured the Torna Fort, marking the beginning of a relentless campaign to liberate his people from foreign oppression.\n      Over the following decades, he utilized innovative guerrilla warfare tactics, known as Ganimi Kava, to outmaneuver the\n       much larger and more pow